<a href="https://colab.research.google.com/github/DURGAPRASAD-67/-ai-mentor-portfolio/blob/main/Day6_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [5]:
# Load 5 sample résumés
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] FAILED: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] FAILED: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 

3/5 succeeded, 2 failed


In [6]:
# Empty string
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"B.Sc. Computer Science","institution":"University of Example","year":2020}],"skills":["Python","Data Analysis","JSON"],"projects":[],"experience_years":2.5}
Whitespace input: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
Garbage input: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.


## Day 6 Lab 6A — Errors handled

1. **Markdown fence wrapping** (` ```json ... ``` `)
   Gemini sometimes returned JSON wrapped in markdown fences.  
   The retry prompt asked the model to return raw JSON only without fences.

2. **Hallucinated phone number when source has none**
   The model occasionally invented a phone number when the résumé did not contain one.  
   Fixed using `Optional[str] = None` in the Pydantic schema so missing values become `null`.

3. **Empty / whitespace-only input**
   Empty input caused validation problems.  
   Input validation checks were added before calling Gemini, and errors are handled gracefully using exceptions.

4. **Hallucination on garbage input**
   Gemini sometimes generated a fake résumé from invalid text like random sentences.  
   Defence: validate input before sending to the model using minimum length checks and email-like pattern checks.

## Technical reasoning

Using structured schema validation with Pydantic is more reliable than only prompting the model to "return JSON".  
The schema ensures:
- missing fields become `null`
- invalid structures raise validation errors
- outputs remain production-safe and predictable

Retry handling and validation improve robustness for real-world AI pipelines.

In [8]:
!pip install google-genai beautifulsoup4 pydantic requests

In [9]:
from google import genai
from pydantic import BaseModel
from typing import List, Optional

import requests
from bs4 import BeautifulSoup

import pathlib
import json

In [11]:
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GOOGLE_API_KEY')
)

In [12]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [13]:
{
  "company": "Amazon",
  "role": "Software Engineer",
  "must_have_skills": ["Python", "DSA"],
  "nice_to_have_skills": ["AWS"],
  "min_cgpa": 7.0,
  "locations": ["Bangalore"],
  "package_lpa": 6.5
}

{'company': 'Amazon',
 'role': 'Software Engineer',
 'must_have_skills': ['Python', 'DSA'],
 'nice_to_have_skills': ['AWS'],
 'min_cgpa': 7.0,
 'locations': ['Bangalore'],
 'package_lpa': 6.5}

In [14]:
def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text."""

    try:
        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        r.raise_for_status()

        soup = BeautifulSoup(r.text, 'html.parser')

        # Remove useless tags
        for tag in soup(['script', 'style']):
            tag.decompose()

        text = soup.get_text(
            separator='\n',
            strip=True
        )

        return text[:max_chars]

    except Exception as e:
        print(f'Scrape failed: {e}')
        return None

In [15]:
test_url = "https://amazon.jobs/en/jobs/10386843/software-dev-engineer-ii"

text = fetch_jd(test_url)

if text:
    print(f"Got {len(text)} chars")
    print(text[:500])
else:
    print("Scraping blocked")

Got 5213 chars
Software Dev Engineer II - Job ID: 10386843 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Accommodations
Benefits
Inclusive experiences
How We Hire
Leadership principles
Working at Amazon
FAQ
Software Dev Engineer II
Job ID: 10386843 | ADCI HYD 13 SEZ
Apply now
Description
The Amazon Business team is looking for candidates who are passionate about delivering an amazing experience to our i


In [16]:
def normalise_jd(text: str) -> JD:

    resp = client.models.generate_content(
        model='gemini-2.5-flash',

        contents=f"""
        Extract a structured job description JSON from this text:

        {text}
        """,

        config={
            'response_mime_type': 'application/json',

            'response_schema': JD.model_json_schema(),
        },
    )

    return JD.model_validate_json(resp.text)

In [17]:
if text:
    jd = normalise_jd(text)

    print(
        jd.model_dump_json(indent=2)
    )

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

In [18]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GOOGLE_API_KEY')
)

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

In [19]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get('GOOGLE_API_KEY')
)

In [20]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


In [21]:
jd = normalise_jd(text)

print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Software Dev Engineer II",
  "must_have_skills": [
    "Software Development (3+ years professional experience)",
    "System Design or Architecture (2+ years non-internship experience)",
    "Design Patterns",
    "Reliability",
    "Scaling",
    "Programming Language (at least one)",
    "Computer Science fundamentals",
    "Ownership",
    "Customer Experience focus",
    "Troubleshooting"
  ],
  "nice_to_have_skills": [
    "Web-based applications development",
    "Web services-based applications development",
    "Software Development Life Cycle (3+ years experience)",
    "Coding standards",
    "Code reviews",
    "Source control management",
    "Build processes",
    "Testing",
    "Operations",
    "Algorithms",
    "Object-oriented design",
    "Distributed systems",
    "Web development",
    "Front-end design",
    "Back-end design",
    "APIs",
    "Micro-Services",
    "Micro-Frontends",
    "SDKs",
    "Bachelor's degree in Computer